In [24]:
import pandas as pd
import glob
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import random

In [25]:
folder_r = r"C:\Arjun\Thesis\data\20200421_170039-sunset1\filtered chunks\un-normalised"
folder_q= r"C:\Arjun\Thesis\data\20200422_172431-sunset2\unormal_subs"
exp_path =r"C:\Arjun\Thesis\results\exp_0\longer"
NUM_MATCHES = 50
MATCH_DIST = 10
ref_length_min = 45
ref_length_max = 240 # Maximum length of reference sequence in seconds
query_length_min =0.4
query_length_max = 1
exp_no = "0_longer"

In [26]:

def unix_to_brisbane(unix_time):
    """Convert UNIX timestamp to Brisbane time (UTC+10)"""
    # Brisbane is UTC+10 (no daylight saving)
    brisbane_time = datetime.fromtimestamp(unix_time, tz=timezone.utc) + pd.Timedelta(hours=10)
    return brisbane_time.strftime('%Y-%m-%d %H:%M:%S')

In [27]:
print(unix_to_brisbane(1587540426.9))
1587705130.8
       

2020-04-22 17:27:06


1587705130.8

In [28]:
print(unix_to_brisbane(1587705303.8))

2020-04-24 15:15:03


In [29]:
def get_timestamps(folder_path):
    """Extract 4th column timestamps from all CSVs in folder"""
    timestamps = []
    for csv_file in glob.glob(f"{folder_path}/*.csv"):
        df = pd.read_csv(csv_file)
        ts_col = df.iloc[:, 3]  # 4th column (0-indexed)
        timestamps.extend(ts_col.dropna().tolist())
    return set(timestamps)

In [30]:
ts_r = get_timestamps(folder_r)
ts_q = get_timestamps(folder_q)



In [31]:
min_r = unix_to_brisbane(min(ts_r))
max_r = unix_to_brisbane(max(ts_r))
min_q = unix_to_brisbane(min(ts_q))
max_q = unix_to_brisbane(max(ts_q))
print(f"Ref - Min: {min_r}, Max: {max_r}")
print(f" Ref {int((max(ts_r) - min(ts_r))/60)} min {int((max(ts_r) - min(ts_r)) % 60)} sec  ") 
print(f"query - Min: {min_q}, Max: {max_q}s")
print(f" query {int((max(ts_q) - min(ts_q))/60)} min {int((max(ts_q) - min(ts_q)) % 60)} sec  ") 

Ref - Min: 2020-04-21 17:03:04, Max: 2020-04-21 17:07:08
 Ref 4 min 3 sec  
query - Min: 2020-04-22 17:24:21, Max: 2020-04-22 17:28:10s
 query 3 min 49 sec  


In [32]:
ref_gps = pd.read_csv(r"C:\Arjun\Thesis\data\20200421_170039-sunset1\sunset1_gps_v2.csv")
query_gps = pd.read_csv(r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv")
print(len(ref_gps))
print(len(query_gps))

578
566


In [33]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth radius in meters
    R = 6371000

    # Convert latitude and longitude from degrees to radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)

    # Calculate differences between latitudes and longitudes
    d_lat = lat2_rad - lat1_rad
    d_lon = lon2_rad - lon1_rad

    # Haversine formula
    a = np.sin(d_lat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(d_lon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    # Calculate the distance
    distance = R * c

    return distance

In [34]:
# Truncate ref_gps and query_gps to the time windows defined by ts_r and ts_q
ref_min_ts = min(ts_r)
ref_max_ts = max(ts_r)
query_min_ts = min(ts_q)
query_max_ts = max(ts_q)

# Filter ref_gps: keep rows whose elapsed_time_ts falls within [ref_min_ts, ref_max_ts]
ref_gps = ref_gps[
    (ref_gps["elapsed_time_ts"] >= ref_min_ts) &
    (ref_gps["elapsed_time_ts"] <= ref_max_ts)
].reset_index(drop=True)

# Filter query_gps: keep rows whose elapsed_time_ts falls within [query_min_ts, query_max_ts]
query_gps = query_gps[
    (query_gps["elapsed_time_ts"] >= query_min_ts) &
    (query_gps["elapsed_time_ts"] <= query_max_ts)
].reset_index(drop=True)

print(f"ref_gps truncated to [{unix_to_brisbane(ref_min_ts)}, {unix_to_brisbane(ref_max_ts)}]: {len(ref_gps)} rows")
print(f"query_gps truncated to [{unix_to_brisbane(query_min_ts)}, {unix_to_brisbane(query_max_ts)}]: {len(query_gps)} rows")

ref_gps truncated to [2020-04-21 17:03:04, 2020-04-21 17:07:08]: 220 rows
query_gps truncated to [2020-04-22 17:24:21, 2020-04-22 17:28:10]: 201 rows


In [35]:
ref_gps.columns

Index(['latitude', 'longitude', 'elapsed_time', 'elapsed_time_ts'], dtype='str')

In [36]:
ref_lat = ref_gps['latitude']
ref_lon = ref_gps['longitude']
ref_ts = ref_gps['elapsed_time_ts']  # timestamp column

query_lat = query_gps['latitude']
query_lon = query_gps['longitude']
query_ts = query_gps['elapsed_time_ts']

'''
# Use GPS elapsed_time_ts instead of CSV timestamps
min_r = unix_to_brisbane(min(ref_ts))
max_r = unix_to_brisbane(max(ref_ts))
min_q = unix_to_brisbane(min(query_ts))
max_q = unix_to_brisbane(max(query_ts))'''
print(f"Ref - Min: {min_r}, Max: {max_r}, period: {(max(ts_r) - min(ts_r))/60} min")
print(f"query - Min: {min_q}, Max: {max_q}, period: {(max(ts_q) - min(ts_q))/60} min")

Ref - Min: 2020-04-21 17:03:04, Max: 2020-04-21 17:07:08, period: 4.060927681128184 min
query - Min: 2020-04-22 17:24:21, Max: 2020-04-22 17:28:10, period: 3.819243899981181 min


In [37]:
print(ref_ts)

0      1.587453e+09
1      1.587453e+09
2      1.587453e+09
3      1.587453e+09
4      1.587453e+09
           ...     
215    1.587453e+09
216    1.587453e+09
217    1.587453e+09
218    1.587453e+09
219    1.587453e+09
Name: elapsed_time_ts, Length: 220, dtype: float64


In [38]:
print(len(ref_lat))

220


In [39]:
window_r = set(ts_r)  # timestamps from first folder's CSVs
window_q = set(ts_q)
#window_r = set(ref_ts)  # timestamps from first folder's CSVs
#window_q = set(query_ts)

In [40]:
def find_nearest_gps_match(ref_lat, ref_lon, ref_ts, query_lat, query_lon, query_ts, window_r, window_q, match_dist):
    """Find the nearest GPS match in query for a single reference point (only if within match_dist meters)"""
    best_match = None   
    for j, (qlat, qlon, qts) in enumerate(zip(query_lat, query_lon, query_ts)):
        dist = haversine_distance(ref_lat, ref_lon, qlat, qlon)
        
        if dist <= match_dist:
            best_match = {
                'ref_lat': ref_lat, 
                'ref_lon': ref_lon,
                'query_lat': qlat, 
                'query_lon': qlon,
                'ref_ts': ref_ts,
                'ref_time': unix_to_brisbane(ref_ts),
                'query_ts': qts,
                'query_time': unix_to_brisbane(qts),
                'distance_m': dist,
            
            }
            return best_match
    
    # Only return match if distance is within threshold

    return None

In [41]:
# Randomly select NUM_MATCHES from reference points
print(f"\nRandomly selecting {NUM_MATCHES} reference points from {len(ref_lat)} total points...")
random.seed(42)  # Set seed for reproducibility (optional)
random_indices = random.sample(range(len(ref_lat)), min(NUM_MATCHES, len(ref_lat)))
print(random_indices)
matches = []
print(f"Finding nearest GPS matches for each selected reference point (threshold: {MATCH_DIST}m)...")
print("-" * 60)

for idx in random_indices:
    print(idx)
    match = find_nearest_gps_match(
        ref_lat.iloc[idx], ref_lon.iloc[idx], ref_ts.iloc[idx],
        query_lat, query_lon, query_ts,
        window_r, window_q, MATCH_DIST
    )
    if match:
        matches.append(match)
        print(f"Reference point {idx}: Distance = {match['distance_m']:.2f}")
    else:
        print(f"Reference point {idx}: No match found within {MATCH_DIST}m")

print("-" * 60)
print(f"COMPLETED! Found {len(matches)} matches within {MATCH_DIST}m threshold")


Randomly selecting 50 reference points from 220 total points...
[163, 28, 6, 189, 70, 62, 57, 35, 188, 26, 173, 216, 139, 22, 151, 108, 8, 7, 23, 55, 59, 129, 154, 217, 143, 50, 183, 166, 179, 207, 107, 56, 114, 150, 71, 1, 40, 178, 204, 87, 185, 39, 200, 86, 210, 201, 97, 24, 91, 88]
Finding nearest GPS matches for each selected reference point (threshold: 10m)...
------------------------------------------------------------
163
Reference point 163: Distance = 5.18
28
Reference point 28: Distance = 0.70
6
Reference point 6: Distance = 0.76
189
Reference point 189: Distance = 3.94
70
Reference point 70: Distance = 3.49
62
Reference point 62: Distance = 4.98
57
Reference point 57: Distance = 4.28
35
Reference point 35: Distance = 2.04
188
Reference point 188: Distance = 3.77
26
Reference point 26: Distance = 0.63
173
Reference point 173: Distance = 0.33
216
Reference point 216: No match found within 10m
139
Reference point 139: Distance = 2.85
22
Reference point 22: Distance = 1.12
151


In [42]:
# Results
if len(matches) > 0:
    matches_df = pd.DataFrame(matches)
   
    print(f"\n--- MATCH SUMMARY ---")
    print(f"Total matches found: {len(matches)}")
    
    print(f"Average match distance: {matches_df['distance_m'].mean():.2f} meters")
    print(f"Min distance: {matches_df['distance_m'].min():.2f} meters")
    print(f"Max distance: {matches_df['distance_m'].max():.2f} meters")
    
    
    # Save all matches to CSV
   
    #matches_df.to_csv(f"exp/gps_matches_{exp_no}.csv", index=False)
    matches_df.to_csv(exp_path +f"/gps_matches_{exp_no}.csv", index=False)
    print(f"\n✓ Saved all {len(matches)} matches to 'gps_matches_{NUM_MATCHES}_samples.csv'")
    
    # Also save only valid matches (timestamps within both windows)
    
    #matches_df.to_csv(f"valid_matches.csv", index=False)
    
else:
    print("No matches found within threshold!")
    print(f"Try increasing MATCH_DIST (currently {MATCH_DIST}m) or check if GPS trajectories overlap")


--- MATCH SUMMARY ---
Total matches found: 47
Average match distance: 3.62 meters
Min distance: 0.33 meters
Max distance: 9.96 meters

✓ Saved all 47 matches to 'gps_matches_50_samples.csv'


In [43]:
# Load your GPS file (adjust path as needed)
'''gps_file = pd.read_csv(r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv")

# Extract coordinates
latitudes = gps_file['latitude']
longitudes = gps_file['longitude']

# Get first coordinate
first_lat = latitudes.iloc[0]
first_lon = longitudes.iloc[0]

# Calculate distances from first point to all others
distances = []
for i in range(len(latitudes)):
    dist = haversine_distance(first_lat, first_lon, latitudes.iloc[i], longitudes.iloc[i])
    distances.append(dist)

# Add distances to dataframe
gps_file['distance_from_first_m'] = distances

# Display results
print(f"First coordinate: ({first_lat}, {first_lon})")
print(f"Total points: {len(distances)}")
print("\nFirst 10 distances from first point:")
for i in range(len(distances)):
    print(f"Point {i}: {distances[i]:.2f} meters")

# Show summary
print(f"\n--- Summary ---")
print(f"Max distance from first point: {max(distances):.2f} meters")
print(f"Min distance from first point: {min(distances):.2f} meters")
print(f"Average distance: {sum(distances)/len(distances):.2f} meters")

# Optional: Save to new CSV
gps_file.to_csv("gps_with_distances_from_first.csv", index=False)
print("\n✓ Saved to 'gps_with_distances_from_first.csv'")'''

<>:2: SyntaxWarning: invalid escape sequence '\A'
<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\arj_s\AppData\Local\Temp\ipykernel_16084\781749876.py:2: SyntaxWarning: invalid escape sequence '\A'
  '''gps_file = pd.read_csv(r"C:\Arjun\Thesis\data\20200422_172431-sunset2\sunset2_gps_v2.csv")


'gps_file = pd.read_csv(r"C:\\Arjun\\Thesis\\data\x8200422_172431-sunset2\\sunset2_gps_v2.csv")\n\n# Extract coordinates\nlatitudes = gps_file[\'latitude\']\nlongitudes = gps_file[\'longitude\']\n\n# Get first coordinate\nfirst_lat = latitudes.iloc[0]\nfirst_lon = longitudes.iloc[0]\n\n# Calculate distances from first point to all others\ndistances = []\nfor i in range(len(latitudes)):\n    dist = haversine_distance(first_lat, first_lon, latitudes.iloc[i], longitudes.iloc[i])\n    distances.append(dist)\n\n# Add distances to dataframe\ngps_file[\'distance_from_first_m\'] = distances\n\n# Display results\nprint(f"First coordinate: ({first_lat}, {first_lon})")\nprint(f"Total points: {len(distances)}")\nprint("\nFirst 10 distances from first point:")\nfor i in range(len(distances)):\n    print(f"Point {i}: {distances[i]:.2f} meters")\n\n# Show summary\nprint(f"\n--- Summary ---")\nprint(f"Max distance from first point: {max(distances):.2f} meters")\nprint(f"Min distance from first point: 

In [44]:
import json
config = {
    "experiment_name": f"dtw_test_{NUM_MATCHES}_{exp_no}",
    "data": {
        "query_folder": folder_q,  # Using your existing folder variables
        "ref_folder": folder_r
    },
    "max_ref_l": ref_length_max,
    "max_query_l":query_length_max,
    "pairs": []
}

In [45]:
#for idx, row in matches_df.iterrows():
for idx, row in matches_df.sample(frac=1).iterrows():
    # Get timestamps
    ref_ts = row['ref_ts']
    query_ts = row['query_ts']
    
    # Randomize window lengths (less than max)
    import random
    ref_len = round(random.uniform(ref_length_min, ref_length_max), 1)
    query_len = round(random.uniform(query_length_min, query_length_max), 1)

    # Calculate window starts (ensure timestamp is not at midpoint)
    # Random offset between 20% and 80% of window length
    ref_offset_pct = random.uniform(0.2, 0.8)
    query_offset_pct = random.uniform(0.2, 0.8)
    
    ref_start = round(ref_ts - (ref_len * ref_offset_pct), 1)
    ref_end = round(ref_ts + (ref_len * (1 - ref_offset_pct)), 1)

    if(ref_start< min(window_r )):
        ref_start = min(window_r )
    
    if(ref_end > max(window_r )):
        ref_end = max(window_r )
    ref_len = ref_end- ref_start

    query_start = round(query_ts - (query_len * query_offset_pct), 1)
    query_end = round(query_ts + (query_len * (1 - query_offset_pct)), 1)

    if(query_start< min(window_q)):
        query_start = min(window_q)
    
    if(query_end > max(window_q)):
        query_end = max(window_q)
    query_len  = query_end - query_start

    # Create pair config
    pair = {
        "pair_id": idx + 1,
        "query_start": query_start,
        "query_end": query_end,
        "ref_start": ref_start,
        "ref_end": ref_end,
        "ref_l": ref_len,
        "query_l": query_len,
        "ref_match": ref_ts,
        "q_match": query_ts,
        "gps_q": [round(row['query_lat'], 6), round(row['query_lon'], 6)],  # GPS to 6 decimals
        "type": "good"
    }
    config["pairs"].append(pair)

#with open(f'exp/{exp_no}_mdtw_config.json', 'w') as f:
with open(exp_path + f'/{exp_no}_mdtw_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f"✓ Generated config with {len(config['pairs'])} pairs")
print(f"Saved to exp/{exp_no}_mdtw_config.json")

✓ Generated config with 47 pairs
Saved to exp/0_longer_mdtw_config.json
